## SIMULATOR AND DEMONSTRATOR TUTORIAL

### SYSTEM INITIALIZATION

#### System_ubrella from zero

In [ ]:
from pathlib import Path
import sys
from importlib import reload

cwd = Path.cwd().resolve()
search_roots = [cwd, cwd.parent]
module_dir = next((root for root in search_roots if (root / "VIVA_UPNA_Sim_Dem.py").exists()), cwd.parent)
resources_dir = next((root / "additional_resources" for root in search_roots if (root / "additional_resources").is_dir()), cwd / "additional_resources")

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import VIVA_UPNA_Sim_Dem as vus
reload(vus)

# We initialize an umbrella system
system_umbrella = vus.System_umbrella()
# We set the configuration with the example JSON file
# This configuration sets exclusively the eye parameters and sensors placement. It does not set any movement, so the eyes will be in
# their primary position
system_umbrella.set_configuration("additional_resources/umbrella_config_example_init.json")

# We produce a movement in the simulator
system_umbrella.simulator.look_at_this_point([0, 0, 500])

# We check that the last_measurement object from both simulator and demonstrator are linked
system_umbrella.simulator.get_last_measurements(static=True)
if system_umbrella.simulator.last_measurement is system_umbrella.demonstrator.last_measurement:
    print('Both last_measurement objects are linked')
else:
    print('Both last_measurement objects are not the same')

# # We show the simulator's situation
# pl = system_umbrella.simulator.show()
# pl.show()
# pl.close()

#### Initializing an umbrella system from a set configuration

In [ ]:
import json

# We force a movement of the eyes to an unusual position
system_umbrella.simulator.look_at_this_point([0, -1000, 500])
system_umbrella.simulator.get_last_measurements(static=True)
# pl = system_umbrella.simulator.show()
# pl.show()
# pl.close()

# We save the configuration of the system
dict_umbrella = system_umbrella.to_dict()
with open("additional_resources/umbrella_config_example_set_config.json", "w") as json_file:
    json.dump(dict_umbrella, json_file, indent=4)


# We now load the configuration in a new umbrella system, checking that the eyes recover the same position and a measurement has been made
new_system_umbrella = vus.System_umbrella()
new_system_umbrella.set_configuration("additional_resources/umbrella_config_example_set_config.json")
# pl = new_system_umbrella.simulator.show()
# pl.show()
# pl.close()

### SIMULATOR AND DEMONSTRATOR INITIALIZATION
#### Simulator initialization

In [ ]:
# We create a simulator system, loading a default configuration
system_simulator = vus.System_simulator()
system_simulator.set_configuration("additional_resources/simulator_config_example_init.json")
# We show the simulator's situation, gazing at a point directly ahead
system_simulator.look_at_this_point([0, 0, 500])
system_simulator.get_last_measurements(static=True)
# pl = system_simulator.show()
# pl.show()
# pl.close()

# We de-activate the noise to get a perfect estimation of the eye position
system_simulator.glasses.noise_sensor_model = 'no_noise'

# We gaze upwards so that the sensors hit the sclera
system_simulator.look_at_this_point([0, -1000, 500])
system_simulator.get_last_measurements(static=True)
# pl = system_simulator.show()
# pl.show()
# pl.close()

# We save the last_measurement of the simulator
last_measurement = system_simulator.last_measurement.copy()

#### Demonstrator initialization

In [ ]:
# We create a demonstrator system, loading the same configuration as the simulator
system_demonstrator = vus.System_demonstrator()
system_demonstrator.set_configuration("additional_resources/demonstrator_config_example_init.json")

# We load the last_measurement of the simulator into the demonstrator to estimate the sclera spheres
system_demonstrator.set_last_measurement(last_measurement)
center_left, radius_left = system_demonstrator.stat_estimate_eye_centers('left', est_method='fixed', fixed_radius=system_simulator.user.eyes['left'].r_sclera)
center_right, radius_right = system_demonstrator.stat_estimate_eye_centers('right', est_method='fixed', fixed_radius=system_simulator.user.eyes['right'].r_sclera)

# We check that the estimated radii are extremely close to the real ones
if abs(radius_left - system_simulator.user.eyes['left'].r_sclera) < 1e-3 and abs(radius_right - system_simulator.user.eyes['right'].r_sclera) < 1e-3:
    print("Estimated radii are extremely close to the real ones")
else:
    print("Estimated radii are not close to the real ones")

# # We show the demonstrator's situation. As there have been no optical axis estimations, the eyes will be show in the default position, but
# # in the right place
# pl = system_demonstrator.show()
# pl.show()
# pl.close()

# We save the configuration of the demonstrator
dict_demonstrator = system_demonstrator.to_dict()
# We save the json file:
with open("additional_resources/demonstrator_config_example_set_config.json", "w") as json_file:
    json.dump(dict_demonstrator, json_file, indent=4)

#### Estimations' persistency

In [ ]:
# We load the configuration in a new demonstrator system, checking that the estimations are saved
new_system_demonstrator = vus.System_demonstrator()
new_system_demonstrator.set_configuration("additional_resources/demonstrator_config_example_set_config.json")
# pl = new_system_demonstrator.show()
# pl.show()
# pl.close()

### STATIC ANALYSIS
##### We simulate an eyeball calibration in order to be able to make estimations

In [ ]:
# We load the configuration in the same demonstrator system, overwriting only the calibration parameters
system_demonstrator.set_configuration("additional_resources/demonstrator_config_example_calib.json")

#### Optical axis estimation

In [ ]:
import numpy as np

# Once we have estimated the eye centers, we can estimate the optical axes (provided the eye is calibrated)
# We produce a favorable position of the eyes and record the measurement
system_simulator.look_at_this_point([0, -100, 500])
system_simulator.get_last_measurements(static=True)
# pl = system_simulator.show()
# pl.show()
# pl.close()
last_measurement = system_simulator.last_measurement.copy()

# We load the measurement onto the demonstrator and perform the estimations
system_demonstrator.set_last_measurement(last_measurement)
cornea_center_left = system_demonstrator.stat_estimate_cornea_center('left')
cornea_center_right = system_demonstrator.stat_estimate_cornea_center('right')

# We get the optical axes from the cornea centers
optical_axis_left = cornea_center_left - np.array(system_demonstrator.estimations.eyes['left'].location)
optical_axis_left = optical_axis_left / np.linalg.norm(optical_axis_left)
optical_axis_right = cornea_center_right - np.array(system_demonstrator.estimations.eyes['right'].location)
optical_axis_right = optical_axis_right / np.linalg.norm(optical_axis_right)

# We check that the estimated optical axes are close to the real ones
real_optical_axis_left = system_simulator.user.eyes['left'].optical_axis
real_optical_axis_right = system_simulator.user.eyes['right'].optical_axis
diff_left = np.arccos(np.dot(optical_axis_left, real_optical_axis_left))
diff_right = np.arccos(np.dot(optical_axis_right, real_optical_axis_right))
if diff_left < np.deg2rad(1) and diff_right < np.deg2rad(1):
    print("Estimated optical axes are close to the real ones")
else:
    print("Estimated optical axes are not close to the real ones")

# We update the optical axes in the demonstrator
system_demonstrator.estimations.eyes['left'].optical_axis = optical_axis_left
system_demonstrator.estimations.eyes['right'].optical_axis = optical_axis_right
# We also update the visual axes, in order to make the plot consistent
system_demonstrator.estimations.eyes['left'].visual_axis = optical_axis_left
system_demonstrator.estimations.eyes['right'].visual_axis = optical_axis_right
# # We show the demonstrator's situation. As the optical axes have been estimated, the eyes will be shown looking in the right direction
# pl = system_demonstrator.show()
# pl.show()
# pl.close()

### DYNAMIC ANALYSIS

In [ ]:
# We make the eyes gaze upwards, so that the sensors hit the sclera
system_simulator.look_at_this_point([500, -1000, 500])
system_simulator.get_last_measurements(static=True)
# pl = system_simulator.show()
# pl.show()
# pl.close()

# We produce a rotation of both eyes around a vertical axis
rot_vector = np.array([0, -np.deg2rad(45), 0])
system_simulator.rotate_eyeball(rot_vector, 'left')
system_simulator.rotate_eyeball(rot_vector, 'right')
system_simulator.get_last_measurements()
# pl = system_simulator.show()
# pl.show()
# pl.close()

# We load the last_measurement onto the demonstrator and we perform an estimation of the rotation
system_demonstrator.set_last_measurement(system_simulator.last_measurement.copy())
est_rot_vector_left = system_demonstrator.estimate_angular_vel('left')
est_rot_vector_right = system_demonstrator.estimate_angular_vel('right')

# We check that the estimated rotations are close to the real ones
if np.linalg.norm(est_rot_vector_left - rot_vector) < 0.01 and np.linalg.norm(est_rot_vector_right - rot_vector) < 0.01:
    print("Estimated rotation vectors are close to the real ones")
else:
    print("Estimated rotation vectors are not close to the real ones")

### SEQUENCIAL ANALYSIS

In [ ]:
import matplotlib.pyplot as plt

# We "close" the pupil in order to maximize the number of sensors hitting the iris, allowing for iris estimation
system_simulator.user.eyes['left'].r_pupil = 0.0
system_simulator.user.eyes['right'].r_pupil = 0.0
system_demonstrator.estimations.eyes['left'].r_pupil = 0.0
system_demonstrator.estimations.eyes['right'].r_pupil = 0.0

# We generate an array of measurements using the simulator's simulate_tertiary_movement function
measurement_list, _ = system_simulator.simulate_tertiary_movement([-5000, 100, 500], [5000, -100, 500], 'constant', 'n_samples', n_samples=500)

# We create the error arrays for the sclera radius and optical axis estimations
error_radius = {'left': [], 'right': []}
error_optaxis = {'left': [], 'right': []}
meas_names = {'left': 'left_case', 'right': 'right_case'}

# We define whether the iris estimation accounts for cornea refraction or not
refraction = True

# We loop through the measurement list
for i in range(len(measurement_list)):

    # We load the measurement onto the demonstrator
    system_demonstrator.set_last_measurement(measurement_list[i])

    # We perform the estimations for both eyes, if possible
    for lor in ['left', 'right']:
        # We check if the case is favorable for any estimation
        if system_demonstrator.meas_case[meas_names[lor]] % 10 >= 4:
            # At least three sensors hit the sclera, so we can perform an estimation of the sclera sphere
            center, radius = system_demonstrator.stat_estimate_eye_centers(lor)

            # We check that the result is close to the real one
            diff = abs(radius - system_simulator.user.eyes[lor].r_sclera)
            error_radius[lor].append(diff)

        elif system_demonstrator.meas_case[meas_names[lor]] % 100 // 10 >= 3:
            # At least three sensors hit the iris, so we can perform an estimation of the iris plane
            # We select whether the iris estimation accounts for cornea refraction or not
            if refraction:
                cornea_center = system_demonstrator.stat_estimate_cornea_center(lor, verbose=0)

                # We get the optical axes from the cornea centers
                optical_axis = cornea_center - np.array(system_demonstrator.estimations.eyes[lor].location)
                optical_axis = optical_axis / np.linalg.norm(optical_axis)
            else:
                # We use the estimate_iris_planes() function
                optical_axis_left, _, optical_axis_right, _ = system_demonstrator.estimate_iris_planes()
                if lor == 'left': optical_axis = optical_axis_left
                else: optical_axis = optical_axis_right
                optical_axis = optical_axis / np.linalg.norm(optical_axis)

            # We save the estimation
            system_demonstrator.estimations.eyes[lor].optical_axis = optical_axis
            system_demonstrator.estimations.eyes[lor].visual_axis = optical_axis

            # We check that the estimated optical axes are close to the real ones
            real_optical_axis = system_simulator.gaze_trajectory.optical_axis_list[lor][-len(measurement_list) + i]
            real_optical_axis = real_optical_axis / np.linalg.norm(real_optical_axis)
            diff = np.rad2deg(np.arccos(np.dot(optical_axis, real_optical_axis)))
            error_optaxis[lor].append(diff)

# We plot the errors in the estimations
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
# We plot the error in the radius estimation, marking the 1e-3 threshold
axs[0, 0].plot(error_radius['left'])
axs[0, 0].plot([0, len(error_radius['left'])], [1e-3, 1e-3], 'r--')

axs[0, 1].plot(error_radius['right'])
axs[0, 1].plot([0, len(error_radius['right'])], [1e-3, 1e-3], 'r--')

# We plot the error in the optical axis estimation, marking the 1 degree threshold
axs[1, 0].plot(error_optaxis['left'])
axs[1, 0].plot([0, len(error_optaxis['left'])], [1, 1], 'r--')

axs[1, 1].plot(error_optaxis['right'])
axs[1, 1].plot([0, len(error_optaxis['right'])], [1, 1], 'r--')
        